### Query Types Analysis
This part is based on the dataset called DL-Hard, which is an annotated dataset buiilt upon TREC Deep Learning questions. The questions are annotated with query intent categories based on the paper An Intent Taxonomy for Questions Asked in Web Search (https://marksanderson.org/files/papers/CHIIR21b.pdf) and answer type which is a manual annotation of target answer type for we questions (short answer, factoid etc)

Here, we try to understand how different query intent and answer type affect the effectiveness of query expansion techniques.

In [1]:
import pyterrier as pt
import pandas as pd


dataset = pt.get_dataset("msmarco_passage")
index = dataset.get_index(variant="terrier_stemmed")

# Load DL-hard annotations
annotations_url = "https://raw.githubusercontent.com/grill-lab/DL-Hard/main/annotations/query/annotations.tsv"
annotations = pd.read_csv(
    annotations_url,
    sep="\t",
    header=None,
    names=["qid", "query", "intent", "answer_type", "topic_domain", "serp_type"],
)
annotations["qid"] = annotations["qid"].astype(str)

# Load TREC DL 2019/2020 qrels. This dataset contains relevance judgements from scale 0 to 3.
dl19 = pt.get_dataset("irds:msmarco-passage/trec-dl-2019")
dl20 = pt.get_dataset("irds:msmarco-passage/trec-dl-2020")
all_qrels = pd.concat([dl19.get_qrels(), dl20.get_qrels()])
all_qrels["qid"] = all_qrels["qid"].astype(str)

# Keep only queries that have both annotations and judgements.
qrels_qids = set(all_qrels["qid"].unique())
topics = annotations[annotations["qid"].isin(qrels_qids)].reset_index(drop=True)
print(f"Queries with both annotations and relevance values: {len(topics)}")
print(topics["intent"].value_counts())
print("\n")
print(topics["answer_type"].value_counts())

Queries with both annotations and relevance values: 97
intent
description     48
list            10
quantity         9
reason           8
attribute        7
entity           5
verification     3
process          3
language         3
weather          1
Name: count, dtype: int64


answer_type
factoid              28
definition           21
long answer          16
list                 10
short description     7
short answer          5
comparison            5
guide                 2
weather               1
Long answer           1
multi-answer          1
Name: count, dtype: int64


In [2]:
# Answer type can be further divided instead of keeping 12 categories
category_map = {
    "factoid": "Direct Fact",
    "short answer": "Direct Fact",
    "weather": "Direct Fact",
    "definition": "Explanatory",
    "short description": "Explanatory",
    "long answer": "Explanatory",
    "Long answer": "Explanatory",
    "list": "List",
    "multi-answer": "List",
    "guide": "Explanatory",
    "comparison": "Explanatory",
}
topics["meta_answer_type"] = topics["answer_type"].map(category_map)
print(topics["meta_answer_type"].value_counts())

meta_answer_type
Explanatory    52
Direct Fact    34
List           11
Name: count, dtype: int64


## Classical QE techniques

In [3]:
import re

def clean_query(q):
    q = re.sub(r"[^\w\s]", " ", str(q))
    q = re.sub(r"\s+", " ", q).strip()
    return q

# Clean queries (else it gives errors in PyTerrier)
topics["query"] = topics["query"].apply(clean_query)

# Initialize BM25 retriever
bm25 = pt.terrier.Retriever(index, wmodel="BM25")

# Initialize classic query expansion models
rm3 = pt.rewrite.RM3(index)
bo1 = pt.rewrite.Bo1QueryExpansion(index)

bm25_rm3 = bm25 >> rm3 >> bm25
bm25_bo1 = bm25 >> bo1 >> bm25

Java started (triggered by Retriever.__init__) and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


## LLM-based QE techniques (HyDE)

In [4]:
import sys
!{sys.executable} -m pip install ollama

In [5]:
import ollama
import pandas as pd
from tqdm import tqdm
import os


file_path = "hyde/query_types.csv"

if os.path.exists(file_path):
    print(f"Loading existing HyDE documents from {file_path}...")
    hyde_data = pd.read_csv(file_path)
    hyde_data["qid"] = hyde_data["qid"].astype(str)
    topics = topics.merge(hyde_data[["qid", "hyde_query"]], on="qid", how="left")
    print(topics.columns.tolist())  
    hyde_topics = topics[["qid", "hyde_query"]]

else:
    print("Generating HyDE documents...")

    def generate_hyde_document(query):
        # Taken from the paper "Precise Zero-Shot Dense Retrieval without Relevance Labels"
        # https://arxiv.org/abs/2212.10496
        prompt = f"""Please write a passage to answer the question
        
        Question: {query}
        
        Passage:"""

        response = ollama.generate(model="llama3.1", prompt=prompt)
        return response["response"]

    # Since local generation takes time, use a progress bar
    tqdm.pandas()
    topics["hyde_doc"] = topics["query"].progress_apply(generate_hyde_document)

    # Standard HyDE combines the original query with the hallucinated document
    topics["hyde_query"] = (topics["query"] + " " + topics["hyde_doc"]).apply(clean_query)

    hyde_topics = topics[["qid", "hyde_query"]]

    hyde_topics.to_csv("topics_with_hyde.csv", index=False)
    print(f"HyDE documents generated and saved to {file_path}")

Loading existing HyDE documents from hyde/query_types.csv...
['qid', 'query', 'intent', 'answer_type', 'topic_domain', 'serp_type', 'meta_answer_type', 'hyde_query']


# Running the experiments

In [6]:
from pyterrier.measures import RR, nDCG, AP, R

if "hyde_query" in hyde_topics.columns:
    hyde_topics = hyde_topics.rename(columns={"hyde_query": "query"})

results_classical = pt.Experiment(
    [bm25, bm25_rm3, bm25_bo1],
    topics[["qid", "query"]],  
    all_qrels,
    eval_metrics=[nDCG @ 10, AP(rel=2), RR(rel=2) @ 10, R(rel=2) @ 100],
    names=["BM25", "RM3", "Bo1"],
    perquery=True,
)

results_hyde = pt.Experiment(
    [bm25],
    hyde_topics[["qid", "query"]],
    all_qrels,
    eval_metrics=[nDCG @ 10, AP(rel=2), RR(rel=2) @ 10, R(rel=2) @ 100],
    names=["HyDE_Llama3"],
    perquery=True,
)

results = pd.concat([results_classical, results_hyde], ignore_index=True)

label_map = topics[["qid", "intent", "meta_answer_type"]]
results = results.merge(label_map, on="qid", how="left")

# Overall results
print("\n=== Overall Results ===")
overall = results.groupby(["name", "measure"])["value"].mean().unstack()
print(overall)

# Results by query intent
print("\n=== nDCG@10 by Query Intent ===")
ndcg_by_intent = (
    results[results["measure"] == "nDCG@10"]
    .groupby(["name", "intent"])["value"]
    .mean()
    .unstack()
)
print(ndcg_by_intent)

# Results by answer type
print("\n=== nDCG@10 by Answer Type ===")
ndcg_by_answer_type = (
    results[results["measure"] == "nDCG@10"]
    .groupby(["name", "meta_answer_type"])["value"]
    .mean()
    .unstack()
)
print(ndcg_by_answer_type)


=== Overall Results ===
measure      AP(rel=2)  R(rel=2)@100  RR(rel=2)@10   nDCG@10
name                                                        
BM25          0.290089      0.541421      0.625749  0.487382
Bo1           0.311664      0.568762      0.618295  0.500851
HyDE_Llama3   0.379404      0.611509      0.732000  0.568181
RM3           0.317579      0.575153      0.628371  0.516954

=== nDCG@10 by Query Intent ===
intent       attribute  description    entity  language      list   process  \
name                                                                          
BM25          0.426033     0.474185  0.464549  0.777663  0.447851  0.636769   
Bo1           0.399186     0.480057  0.445410  0.780760  0.503148  0.669123   
HyDE_Llama3   0.752860     0.567917  0.738413  0.418090  0.491407  0.567692   
RM3           0.453578     0.504873  0.459246  0.779319  0.503527  0.665545   

intent       quantity    reason  verification   weather  
name                                       

### Inspecting the queries 

In [7]:
for answer_type in annotations["answer_type"].unique():
    examples = (
        annotations[annotations["answer_type"] == answer_type]["query"].head(3).tolist()
    )
    count = (annotations["answer_type"] == answer_type).sum()
    print(f"\n{str(answer_type).upper()} ({count} queries):")
    for ex in examples:
        print(f"  - {ex}")


LONG ANSWER (37 queries):
  - benefit policy in layoff
  - what is onboarding for credit unions
  - how much money do motivational speakers make

SHORT ANSWER (30 queries):
  - what is an aml surveillance analyst
  - who owns barnhart crane
  - how much does talent directors get paid a year

COMPARISON (5 queries):
  - difference between a company's strategy and business model is
  - difference between a mcdouble and a double cheeseburger
  - difference between rn and bsn

LIST (30 queries):
  - what types of food can you cook sous vide
  - what is supplemental security income used for
  - foods to detox liver naturally

FACTOID (81 queries):
  - average salary for dental hygienist in nebraska
  - how long does it take to get refund from petsmart
  - how much weight on usps letter

SHORT DESCRIPTION (8 queries):
  - who is aziz hashim
  - why did the ancient egyptians call their land kemet, or black land?
  - what type of conflict does della face in o, henry the gift of the magi

DEFI

In [8]:
for intent_type in annotations["intent"].unique():
    examples = (
        annotations[annotations["intent"] == intent_type]["query"].head(3).tolist()
    )
    count = (annotations["intent"] == intent_type).sum()
    print(f"\n{intent_type.upper()} ({count} queries):")
    for ex in examples:
        print(f"  - {ex}")


DESCRIPTION (157 queries):
  - benefit policy in layoff
  - what is onboarding for credit unions
  - what is an aml surveillance analyst

ENTITY (30 queries):
  - who owns barnhart crane
  - who sings monk theme song
  - what language is craith filmed in?

LIST (48 queries):
  - what types of food can you cook sous vide
  - how to get a free xbox one card
  - where is the show shameless filmed

QUANTITY (52 queries):
  - average salary for dental hygienist in nebraska
  - how long does it take to get refund from petsmart
  - how much does talent directors get paid a year

ATTRIBUTE (25 queries):
  - what is reba mcentire's net worth
  - barclays fca number
  - routing number for savings bank of maine

TEMPORAL (6 queries):
  - what year did knee deep come out funkadelic
  - landfill hours
  - when is the evening

PROCESS (9 queries):
  - what temperature and humidity to dry sausage
  - how do they do open heart surgery
  - what the best way to get clothes white

LOCATION (12 queries):

In [9]:
hard_topics = pd.read_csv(
    "https://raw.githubusercontent.com/grill-lab/DL-Hard/main/dataset/topics.tsv",
    sep="\t",
    header=None,
    names=["qid", "query"]
)
hard_qids = set(hard_topics["qid"].astype(str))
dl_hard = pt.get_dataset("irds:msmarco-passage/trec-dl-hard")
hard_qrels = dl_hard.get_qrels()
hard_qrels["qid"] = hard_qrels["qid"].astype(str)

print(hard_qrels.shape)
print(hard_qrels["qid"].nunique()) 

all_qrels = pd.concat([dl19.get_qrels(), dl20.get_qrels(), hard_qrels])
all_qrels["qid"] = all_qrels["qid"].astype(str)

qrels_qids = set(all_qrels["qid"].unique())
topics = annotations[annotations["qid"].isin(qrels_qids)].reset_index(drop=True)

hard_qids = set(hard_topics["qid"].astype(str))
topics["is_hard"] = topics["qid"].isin(hard_qids)

print(f"Hard queries: {topics['is_hard'].sum()}")  


(4256, 4)
50
Hard queries: 50


In [10]:
from pyterrier.measures import RR, nDCG, AP, R

if "hyde_query" in hyde_topics.columns:
    hyde_topics = hyde_topics.rename(columns={"hyde_query": "query"})

# Split topics into hard and non-hard
hard_topics_subset    = topics[topics["is_hard"]][["qid", "query"]].copy()
nonhard_topics_subset = topics[~topics["is_hard"]][["qid", "query"]].copy()

hard_topics_subset["query"]    = hard_topics_subset["query"].apply(clean_query)
nonhard_topics_subset["query"] = nonhard_topics_subset["query"].apply(clean_query)

dl_hard_qrels = hard_qrels.copy()
dl_hard_qrels["qid"] = dl_hard_qrels["qid"].astype(str)

nonhard_qrels = pd.concat([dl19.get_qrels(), dl20.get_qrels()])
nonhard_qrels["qid"] = nonhard_qrels["qid"].astype(str)
nonhard_qrels = nonhard_qrels[nonhard_qrels["qid"].isin(set(nonhard_topics_subset["qid"]))]

# Run experiments separately
def run_experiment(topics_df, qrels_df, hyde_df, label):
    classical = pt.Experiment(
        [bm25, bm25_rm3, bm25_bo1],
        topics_df,
        qrels_df,
        eval_metrics=[nDCG @ 10, AP(rel=2), RR(rel=2) @ 10, R(rel=2) @ 100],
        names=["BM25", "RM3", "Bo1"],
        perquery=True,
    )
    hyde_subset = hyde_df[hyde_df["qid"].isin(set(topics_df["qid"]))]
    hyde_res = pt.Experiment(
        [bm25],
        hyde_subset,
        qrels_df,
        eval_metrics=[nDCG @ 10, AP(rel=2), RR(rel=2) @ 10, R(rel=2) @ 100],
        names=["HyDE_Llama3"],
        perquery=True,
    )
    combined = pd.concat([classical, hyde_res], ignore_index=True)
    combined["group"] = label
    return combined

hard_results    = run_experiment(hard_topics_subset,    dl_hard_qrels,  hyde_topics, "Hard")
nonhard_results = run_experiment(nonhard_topics_subset, nonhard_qrels,  hyde_topics, "Non-Hard")

all_results = pd.concat([hard_results, nonhard_results], ignore_index=True)

print("\n=== Results: Hard vs. Non-Hard ===")
pivot = all_results.pivot_table(index="name", columns=["measure", "group"], values="value")
print(pivot.to_string())


=== Results: Hard vs. Non-Hard ===
measure     AP(rel=2)           R(rel=2)@100           RR(rel=2)@10             nDCG@10          
group            Hard  Non-Hard         Hard  Non-Hard         Hard  Non-Hard      Hard  Non-Hard
name                                                                                             
BM25         0.147106  0.313310     0.454211  0.572496     0.415056  0.598543  0.274333  0.498014
Bo1          0.158566  0.336298     0.450608  0.609229     0.398722  0.592502  0.266630  0.518096
HyDE_Llama3  0.273749  0.408598     0.520985  0.636522     0.649660  0.754751  0.433938  0.605275
RM3          0.164574  0.341393     0.478356  0.608999     0.409579  0.605582  0.284258  0.529642


In [11]:
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

results = []

methods = all_results["name"].unique()
measures = all_results["measure"].unique()

for method in methods:
    for measure in measures:
        subset = all_results[
            (all_results["name"] == method) &
            (all_results["measure"] == measure)
        ]

        hard_scores = subset[subset["group"] == "Hard"]["value"]
        nonhard_scores = subset[subset["group"] == "Non-Hard"]["value"]

        stat, p = mannwhitneyu(hard_scores, nonhard_scores, alternative='two-sided')

        results.append({
            "method": method,
            "measure": measure,
            "p_value": p,
            "hard_mean": hard_scores.mean(),
            "nonhard_mean": nonhard_scores.mean(),
        })

import pandas as pd
results_df = pd.DataFrame(results)

print("\n=== Mann–Whitney U Test Results ===")
print(results_df.sort_values("p_value"))



=== Mann–Whitney U Test Results ===
         method       measure   p_value  hard_mean  nonhard_mean
6           Bo1       nDCG@10  0.000007   0.266630      0.518096
10          RM3       nDCG@10  0.000013   0.284258      0.529642
2          BM25       nDCG@10  0.000020   0.274333      0.498014
4           Bo1     AP(rel=2)  0.000022   0.158566      0.336298
8           RM3     AP(rel=2)  0.000055   0.164574      0.341393
0          BM25     AP(rel=2)  0.000111   0.147106      0.313310
11          RM3  RR(rel=2)@10  0.006316   0.409579      0.605582
7           Bo1  RR(rel=2)@10  0.008142   0.398722      0.592502
3          BM25  RR(rel=2)@10  0.008423   0.415056      0.598543
14  HyDE_Llama3       nDCG@10  0.009163   0.433938      0.605275
5           Bo1  R(rel=2)@100  0.015573   0.450608      0.609229
12  HyDE_Llama3     AP(rel=2)  0.042587   0.273749      0.408598
9           RM3  R(rel=2)@100  0.044432   0.478356      0.608999
1          BM25  R(rel=2)@100  0.051391   0.454211   

In [12]:
# Multiple testing correction using Holm-Bonferroni method

reject, pvals_corrected, _, _ = multipletests(
    results_df["p_value"], method="holm"
)

results_df["p_corrected"] = pvals_corrected
results_df["significant"] = reject

print(results_df.sort_values("p_corrected"))

         method       measure   p_value  hard_mean  nonhard_mean  p_corrected  \
6           Bo1       nDCG@10  0.000007   0.266630      0.518096     0.000112   
10          RM3       nDCG@10  0.000013   0.284258      0.529642     0.000201   
2          BM25       nDCG@10  0.000020   0.274333      0.498014     0.000276   
4           Bo1     AP(rel=2)  0.000022   0.158566      0.336298     0.000283   
8           RM3     AP(rel=2)  0.000055   0.164574      0.341393     0.000664   
0          BM25     AP(rel=2)  0.000111   0.147106      0.313310     0.001224   
11          RM3  RR(rel=2)@10  0.006316   0.409579      0.605582     0.063164   
3          BM25  RR(rel=2)@10  0.008423   0.415056      0.598543     0.073279   
7           Bo1  RR(rel=2)@10  0.008142   0.398722      0.592502     0.073279   
14  HyDE_Llama3       nDCG@10  0.009163   0.433938      0.605275     0.073279   
5           Bo1  R(rel=2)@100  0.015573   0.450608      0.609229     0.093438   
1          BM25  R(rel=2)@10